In [1]:
import pandas as pd
import numpy as np

print("=== ETL PIPELINE INTEGRATION & VALIDATION ===")

# 1. Simulación de carga de datasets
dengue_data = {
    'FECHA_SIGN_SINTOMAS': ['2024-01-15', '2024-01-18', '2024-02-01'],
    'ENTIDAD_RES': [30, 30, 12],
    'MUNICIPIO_RES': [1, 193, 5],
    'ESTATUS_CASO': [2, 2, 2]
}

climate_data = {
    'CVE_GEO': ['30001', '30193', '12005'],
    'EPI_YEAR': [2024, 2024, 2024],
    'EPI_WEEK': [3, 3, 5],
    'PRECIPITATION_MM': [45.2, 12.0, 85.1],
    'TEMP_CELSIUS': [26.5, 28.1, 24.0]
}

df_dengue = pd.DataFrame(dengue_data)
df_climate = pd.DataFrame(climate_data)

# 2. Transformación de datos clínicos
df_dengue['FECHA_SIGN_SINTOMAS'] = pd.to_datetime(df_dengue['FECHA_SIGN_SINTOMAS'])
df_dengue['EPI_YEAR'] = df_dengue['FECHA_SIGN_SINTOMAS'].dt.isocalendar().year
df_dengue['EPI_WEEK'] = df_dengue['FECHA_SIGN_SINTOMAS'].dt.isocalendar().week
df_dengue['CVE_GEO'] = (
    df_dengue['ENTIDAD_RES'].astype(str).str.zfill(2) + 
    df_dengue['MUNICIPIO_RES'].astype(str).str.zfill(3)
)

# Agrupación por semana epidemiológica y municipio
weekly_cases = df_dengue.groupby(['CVE_GEO', 'EPI_YEAR', 'EPI_WEEK']).size().reset_index(name='CONFIRMED_CASES')

# 3. Integración con clima (Inner Join)
df_integrated = pd.merge(
    weekly_cases,
    df_climate,
    on=['CVE_GEO', 'EPI_YEAR', 'EPI_WEEK'],
    how='inner'
)

# 4. Guardar resultado en formato Parquet
output_path = '../data/processed/dengue_climate_integrated.parquet'
df_integrated.to_parquet(output_path, index=False)

print("\n✔ ETL Pipeline executed successfully!")
print(f"✔ File generated at: {output_path}")
print("\n--- Integrated Parquet Dataset Preview ---")
print(df_integrated)

=== ETL PIPELINE INTEGRATION & VALIDATION ===

✔ ETL Pipeline executed successfully!
✔ File generated at: ../data/processed/dengue_climate_integrated.parquet

--- Integrated Parquet Dataset Preview ---
  CVE_GEO  EPI_YEAR  EPI_WEEK  CONFIRMED_CASES  PRECIPITATION_MM  TEMP_CELSIUS
0   12005      2024         5                1              85.1          24.0
1   30001      2024         3                1              45.2          26.5
2   30193      2024         3                1              12.0          28.1
